# Advanced Artificial Intelligence Task 2
### Produce classification via pre-trained models across different architectures

- **CNN**:          EfficientNet_V2
- **TRANSFORMER**:   Swin
- **HYBRID**:       MaxVit
Details for the pre-trained weights can be found [here](https://docs.pytorch.org/vision/stable/models/generated/torchvision.models.efficientnet_v2_s.html#torchvision.models.efficientnet_v2_s).

Training metrics are logged to [Weights & Biases](https://wandb.ai). Before the first run: `wandb login` (one-time) to log into the shared workspace.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import torch.optim as optim
import torchmetrics
import os
from torchvision.transforms import v2 as T
from torchvision.models import get_model, get_weight
from torch.nn import CrossEntropyLoss
from torch.optim import SGD, Adam
from torchmetrics import Metric
import wandb
from tqdm import tqdm
from pathlib import Path
import datetime
import sys
import numpy as np
from collections import Counter
from PIL import Image
import matplotlib.pyplot as plt
from torchvision.transforms import v2 as T
import random
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import train_test_split
from safetensors.torch import save_file
import platform
sys.path.append("..")
sys.path.append(".")
# from experiment_configs import task_2_config as experiments
from experiment_configs import task_2_config_final as experiments
from utils.dataset import ProduceDataset
from utils.mtl_model import MultiTaskClassifier
from utils.utils import seed_everything
from utils.calibration import create_calibration_plots

torch.cuda.empty_cache()
# Set workers based on platform
if sys.platform in ["linux", "darwin"]:
    # Linux, WSL or Mac
    print("Running on a Unix")
    WORKERS = int(os.cpu_count() * 0.75) 
else:
    sys.platform == "win32"
    # Native Windows - seemingly has sensitivity with workers.
    print("Running on Windows")
    WORKERS = int(os.cpu_count() * 0.20) 

In [ ]:
# Papermill parameters - overridden by run_all.py
# Ensure that cell is tagged as 'parameters'
# See run_all.py for batch execution via papermill.
RUN_ALL = "" # LEAVE EMPTY

# === DEFINE EXPERIMENT & DATASET HERE FOR SINGLE RUNS ===
EXP_NAME_DEFAULT = ""
DATASET_PATH = "Fruit_And_Vegetable_Diseases_Dataset_no_identical_no_aug"


In [ ]:
# Resolve EXP after papermill has injected overrides above.
valid_experiments = [
    name for name in dir(experiments)
    if isinstance(getattr(experiments, name), experiments.Experiment)
]
print(f"Selectable experiments:\n {valid_experiments}")

EXP = getattr(experiments, RUN_ALL or EXP_NAME_DEFAULT)
print(f"\nSelected: {EXP.display_name}")


In [ ]:
# TRAINING PARAMETERS
SKIP_TRAIN = False
RECORD_WANDB = True
# Define primary dataset path and validate file count
PRODUCE_DATASET_PATH = Path("data") / DATASET_PATH
EXPECTED_FILES = 19392 
actual = sum(1 for p in PRODUCE_DATASET_PATH.rglob("*") if p.is_file())
assert actual == EXPECTED_FILES, f"Dataset drift: expected {EXPECTED_FILES} files, got {actual}"

# Seed for reproducability
# https://gist.github.com/ihoromi4/b681a9088f348942b01711f251e5f964
seed_everything(42)
gen = torch.Generator()
gen.manual_seed(42)

# Training Set Augmentation
AUGMENT = True if EXP.aug_magnitude > 0 else False
# Augmentation pipeline 
gpu_augment = T.Compose([
    T.RandomHorizontalFlip(),
    T.RandomVerticalFlip(),
    T.RandAugment(num_ops=1, magnitude= EXP.aug_magnitude),
])
print(AUGMENT, gpu_augment)

# MTL WEIGHTS
# Task weighting follows a zero-sum logic to maintain loss scale consistency.
# Primary: Binary Health (Healthy/Rotten)
# Auxiliary: Multiclass Produce Type
# Unused when EXP.is_mtl is False (STL path skips the type head entirely).
TYPE_LOSS_WEIGHT = 1 - EXP.primary_task_weight
assert EXP.primary_task_weight + TYPE_LOSS_WEIGHT == 1

# Batch size of 32 for all experiemnts
# Change batch size and acc proportion as required for performance (in configs)
BATCH_SIZE = EXP.batch_size
ACCUMULATION_STEPS = EXP.acc_steps
EFFECTIVE_BATCH_SIZE = BATCH_SIZE * ACCUMULATION_STEPS
assert EFFECTIVE_BATCH_SIZE == 32, ("Effective batch size must be 32 for training stability. "
                                    "Adjust BATCH_SIZE or GRADIENT_ACCUMULATION_STEPS accordingly.")
TEST_SPLIT = 0.8 
MAXIMUM_EPOCHS = 20 # Define maximum for early-stopping.
NUM_CLASSES = 2 # Healthy & Rotten
EARLY_STOPPING_PATIENCE = 5 # Defines how many non-improvement epochs will terminate run

# Test over-confidence
TEST_CONFIDENCE = True

# Load the pre-trained weights if specified
pretrained_weights = get_weight(EXP.weight_string)
weights_arg = pretrained_weights if EXP.pretrained else None
PRETRAINED_MODEL = get_model(EXP.architecture, weights=weights_arg)
# Extract required transformations for model image input
auto_transforms = pretrained_weights.transforms()
if not EXP.pretrained:
    print(f"Random-init: skipping {EXP.weight_string}")
# print(f"{EXP.weight_string} transforms:\n\n {pretrained_weights.transforms()}")

In [ ]:
# Initialize the dataset and assign labels based on folder structure
produce_dataset = ProduceDataset(dataset_root_dir=PRODUCE_DATASET_PATH,
                                  transform=auto_transforms)
produce_dataset.print_class_balance()
produce_dataset.display_examples(num_samples=5, show_transformed=False)


In [ ]:
# Set transfer learning method
# https://medium.com/@marklpd/transfer-learning-finetuning-for-cnns-in-pytorch-5c4ade873d93
if EXP.training.transfer_type == "FREEZE":
    # Freeze backbone if enabled; gradients are enabled by default for new layers
    for parameters in PRETRAINED_MODEL.parameters():
        # Do not compute backbone gradients (i.e., freeze weights)
        parameters.requires_grad = False 

In [ ]:
# Instantiate model with arhcitecture-specific head replacement logic
# MTL: wrap backbone with MultiTaskClassifier (health + type heads)
# STL: swap the pretrained final layer for a binary (Healthy/Rotten) Linear
if EXP.is_mtl:
    model = MultiTaskClassifier(PRETRAINED_MODEL, num_produce_classes=produce_dataset.num_produce_types)
else:
    # STL: Replace final layer with binary classifier (Healthy/Rotten)
    head_attr = "classifier" if hasattr(PRETRAINED_MODEL, "classifier") else "head"
    head = getattr(PRETRAINED_MODEL, head_attr)
    # Per-arch head replacement: EfficientNet/MaxViT expose .classifier (Sequential => Linear);
    if isinstance(head, nn.Sequential):
        head[-1] = nn.Linear(head[-1].in_features, NUM_CLASSES)
    # Swin exposes .head (single Linear).
    elif isinstance(head, nn.Linear):
        setattr(PRETRAINED_MODEL, head_attr, nn.Linear(head.in_features, NUM_CLASSES))
    else:
        raise ValueError(f"Unsupported STL classification_head: {type(head).__name__}")
    model = PRETRAINED_MODEL

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

In [ ]:
# Extract parameters based on transfer learning method
if EXP.training.transfer_type == "FREEZE":
    # Full backbone freeze
    trainable_params = [parameter for parameter in model.parameters() 
                        if parameter.requires_grad]
else:
    # Freeze none
    print(f"FINETUNE: unfroze all parameters of {EXP.architecture}")
    trainable_params = model.parameters()

In [ ]:
# Define optimiser
# SGD - classic default choice for CNNs.
# AdamW - generally more robust to unoptimal hyperparameters and converges faster
# have more complex gradient landscapes that benefit from adaptive learning rates
if EXP.optimizer == "sgd":
    optimizer = optim.SGD(trainable_params, lr=EXP.training.learning_rate, momentum=EXP.training.momentum)
elif EXP.optimizer == "adamw":
    optimizer = optim.AdamW(trainable_params, lr=EXP.training.learning_rate, weight_decay=0.01)
else: 
    raise ValueError(f"Unsupported optimizer: {EXP.optimizer}")

In [ ]:
# General transfer learning scheduler:
# Cosine decay with a linear warmup
# epoch:  0------1------2------3------..........20
#         |   warmup    |   cosine decay         |
WARM_UP_EPOCHS = 2
# Linearly warm up from 0 to LR to training lr over WARMUP_EPOCHS
warmup = torch.optim.lr_scheduler.LinearLR(
    optimizer, start_factor=0.01, total_iters=WARM_UP_EPOCHS)
# Cosine decay from base LR to 0 across remaining epochs
cosine = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=MAXIMUM_EPOCHS - WARM_UP_EPOCHS)
# Define a sequential scheduler that first applies warmup, then cosine decay
lr_scheduler = torch.optim.lr_scheduler.SequentialLR(
    optimizer, schedulers=[warmup, cosine], milestones=[WARM_UP_EPOCHS])

In [ ]:
# Stratify by produce category + label so the split preserves per-category class balance
# Construct list of of (<category>,<label>) (e.g. (0, 1), (0, 1), (1, 1),...)
stratify_keys = list(zip(produce_dataset.produce_type_lbls,
                         produce_dataset.health_lbls))

train_index, val_index = train_test_split(
    range(len(produce_dataset)), # Split the indices (0 - 29291), not the data itself
    test_size=1 - TEST_SPLIT,
    stratify=stratify_keys,
    random_state=42,
)

train_dataset = torch.utils.data.Subset(produce_dataset, train_index)
val_dataset   = torch.utils.data.Subset(produce_dataset, val_index)

# Assert that the datasets are balanced across produce and health labels
full  = Counter(stratify_keys)
train = Counter(stratify_keys[i] for i in train_index)
val   = Counter(stratify_keys[i] for i in val_index)
TOLERANCE = 0.005
for k in sorted(full):
    full_pct = full[k]/len(stratify_keys)
    train_pct = train[k]/len(train_index)
    val_pct = val[k]/len(val_index)
    status = "GOOD" if abs(train_pct - full_pct) < TOLERANCE and abs(val_pct - full_pct) < TOLERANCE else "BAD"
    print(f"{str(k):<10} full {full_pct:.3f}  train {train_pct:.3f}  val {val_pct:.3f}  {status}")
    assert abs(train_pct - full_pct) < TOLERANCE and abs(val_pct - full_pct) < TOLERANCE, \
        f"Stratification skewed for {k}"

In [ ]:
# Class-weighted loss to counteract Healthy/Rotten & produce imbalances (computed on training set only)
train_health_labels = [produce_dataset.health_lbls[i] for i in train_index]
train_type_labels = [produce_dataset.produce_type_lbls[i] for i in train_index]

def weighted_cross_entropy(labels, indices):
    y = [labels[i] for i in indices]
    weights = compute_class_weight(class_weight='balanced', classes=np.unique(y), y=y)
    return nn.CrossEntropyLoss(weight=torch.tensor(weights, dtype=torch.float).to(device)), weights

if EXP.class_weighted:
    health_criterion, health_weights = weighted_cross_entropy(produce_dataset.health_lbls, train_index)
    type_criterion,   type_weights   = weighted_cross_entropy(produce_dataset.produce_type_lbls, train_index)
    print(f"Health class weights: {health_weights}")
    print(f"Type class weights:   {type_weights}")
else:
    health_criterion = nn.CrossEntropyLoss()
    type_criterion   = nn.CrossEntropyLoss()
    print("Unweighted CE (class_weighted=False)")


In [ ]:
# Initialise dataloaders - https://www.geeksforgeeks.org/deep-learning/pytorch-dataloader/
# Shuffle training data for better generalization
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, 
                          num_workers=WORKERS, pin_memory=True, generator=gen,
                          persistent_workers=True) 
# No need to shuffle validation data 
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, 
                        num_workers=WORKERS, pin_memory=True,
                        persistent_workers=True) 



print(f"\nTraining size: {len(train_dataset)}  |  Validation size: {len(val_dataset)}")

In [ ]:
# Define train/validation functions
# https://medium.com/@ebimsv/mastering-cnns-in-pytorch-week-2-building-and-training-custom-and-pretrained-cnns-for-image-f040572c73c1

class AverageMeter:
    def __init__(self):
        self.reset()

    def reset(self):
        self.sum = 0
        self.count = 0

    def update(self, value, n=1):
        self.sum += value * n
        self.count += n

    @property
    def avg(self):
        return self.sum / self.count if self.count > 0 else 0

def train_one_epoch(model: nn.Module, dataloader: DataLoader,
                    health_criterion: CrossEntropyLoss, type_criterion: CrossEntropyLoss,
                    optimizer: SGD | Adam, device: torch.device, epoch: int,
                    health_accuracy: Metric, type_accuracy: Metric, is_mtl: bool,
                    augment=None ) -> tuple:
    model.train()
    loss_meter = AverageMeter()
    health_loss_meter = AverageMeter()
    type_loss_meter = AverageMeter()
    health_accuracy.reset()
    type_accuracy.reset()
    progress_bar = tqdm(dataloader, desc=f"Epoch {epoch+1} [Training]", leave=False)

    # zero gradients once at the start of the epoch
    optimizer.zero_grad()

    # Iterate over batches (image, health label, type label)
    for id, (X_batch, y_health, y_type) in enumerate(progress_bar):
        X_batch = X_batch.to(device)
        y_health = y_health.to(device)
        
        if augment is not None:
            X_batch = augment(X_batch)
            X_batch = X_batch.clamp(-3.0, 3.0)
        # Forward + loss: MTL returns (health, type); STL returns health only
        if is_mtl:
            y_type = y_type.to(device)
            health_output, type_output = model(X_batch)
            loss_health = health_criterion(health_output, y_health)
            loss_type = type_criterion(type_output, y_type)
            loss = (EXP.primary_task_weight * loss_health
                + TYPE_LOSS_WEIGHT * loss_type)
        else:
            health_output = model(X_batch)
            loss_health = health_criterion(health_output, y_health)
            loss = loss_health

        # Scale loss by accumulation steps so gradients are averaged, not summed
        (loss / ACCUMULATION_STEPS).backward()

        # Only step the optimizer every ACCUMULATION_STEPS batches (or on the last batch)
        if (id + 1) % ACCUMULATION_STEPS == 0 or (id + 1) == len(dataloader):
            optimizer.step()
            optimizer.zero_grad()

        # Update metrics (use unscaled loss for logging)
        loss_meter.update(loss.item(), X_batch.size(0))
        health_loss_meter.update(loss_health.item(), X_batch.size(0))
        health_accuracy.update(health_output.argmax(dim=1), y_health)
        if is_mtl:
            type_loss_meter.update(loss_type.item(), X_batch.size(0))
            type_accuracy.update(type_output.argmax(dim=1), y_type)
        if id % 50 == 0:
            postfix = {"loss": loss_meter.avg, "health_acc": health_accuracy.compute().item()}
            if is_mtl:
                postfix["type_acc"] = type_accuracy.compute().item()
            progress_bar.set_postfix(**postfix)

    avg_loss = loss_meter.avg
    avg_health_loss = health_loss_meter.avg
    avg_type_loss = type_loss_meter.avg if is_mtl else 0.0
    avg_health_acc = health_accuracy.compute().item()
    avg_type_acc = type_accuracy.compute().item() if is_mtl else 0.0

    return avg_loss, avg_health_loss, avg_type_loss, avg_health_acc, avg_type_acc

def validate(model: nn.Module, dataloader: DataLoader, health_criterion: CrossEntropyLoss,
             type_criterion: CrossEntropyLoss, device: torch.device, epoch: int,
             health_accuracy: Metric, type_accuracy: Metric, is_mtl: bool) -> tuple:
    model.eval()
    loss_meter = AverageMeter()
    health_loss_meter = AverageMeter()
    type_loss_meter = AverageMeter()
    health_accuracy.reset()
    type_accuracy.reset()
    progress_bar = tqdm(dataloader, desc=f"Epoch {epoch+1} [Validation]", leave=False)

    with torch.no_grad():
        for id, (X_batch, y_health, y_type) in enumerate(progress_bar):
            X_batch = X_batch.to(device)
            y_health = y_health.to(device)

            if is_mtl:
                y_type = y_type.to(device)
                health_output, type_output = model(X_batch)
                loss_health = health_criterion(health_output, y_health)
                loss_type = type_criterion(type_output, y_type)
                loss = (EXP.primary_task_weight * loss_health
                    + TYPE_LOSS_WEIGHT * loss_type)
            else:
                health_output = model(X_batch)
                loss_health = health_criterion(health_output, y_health)
                loss = loss_health

            loss_meter.update(loss.item(), X_batch.size(0))
            health_loss_meter.update(loss_health.item(), X_batch.size(0))
            health_accuracy.update(health_output.argmax(dim=1), y_health)
            if is_mtl:
                type_loss_meter.update(loss_type.item(), X_batch.size(0))
                type_accuracy.update(type_output.argmax(dim=1), y_type)

            if id % 50 == 0:
                postfix = {"loss": loss_meter.avg, "health_acc": health_accuracy.compute().item()}
                if is_mtl:
                    postfix["type_acc"] = type_accuracy.compute().item()
                progress_bar.set_postfix(**postfix)

    avg_loss = loss_meter.avg
    avg_health_loss = health_loss_meter.avg
    avg_type_loss = type_loss_meter.avg if is_mtl else 0.0
    avg_health_acc = health_accuracy.compute().item()
    avg_type_acc = type_accuracy.compute().item() if is_mtl else 0.0

    return avg_loss, avg_health_loss, avg_type_loss, avg_health_acc, avg_type_acc


In [ ]:
# ── Augmentation visualisation ───────────────────────────────────────────────
# Run this cell to inspect what the model actually sees after augmentation.
# Uses raw PIL images (no normalisation) so colours are true to the original.
if AUGMENT:
    raw_paths = random.sample(produce_dataset.image_paths, 4)
    to_tensor = T.Compose([T.ToImage(), T.ToDtype(torch.float32, scale=True), T.Resize((384, 384))])

    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    fig.suptitle("Top: original  |  Bottom: after augmentation", fontsize=11)

    for i, path in enumerate(raw_paths):
        img    = Image.open(path).convert("RGB")
        tensor = to_tensor(img)

        axes[0, i].imshow(tensor.permute(1, 2, 0).numpy())
        axes[0, i].set_title(Path(path).parent.name, fontsize=7)
        axes[0, i].axis("off")

        augmented = gpu_augment(tensor.unsqueeze(0)).squeeze(0)
        axes[1, i].imshow(augmented.clamp(0, 1).permute(1, 2, 0).numpy())
        axes[1, i].axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
# TRAIN/VALIDATION LOOP
timestamp = datetime.datetime.now().strftime("%Y%m%d%H%M%S")
model_save_name = f"{EXP.display_name }_{timestamp}"
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
torch.cuda.empty_cache()

if not SKIP_TRAIN:
    # View runs at https://wandb.ai/jaimespencer2-/AAI-TASK-2
    if RECORD_WANDB:
        run = wandb.init(
            project="AAI-TASK-2",
            entity="jaimespencer2-",
            name=model_save_name,
            group=EXP.display_name,
            tags=[EXP.architecture, EXP.training.transfer_type, "mtl" if EXP.is_mtl else "stl", platform.node()],
            config={
                "architecture": EXP.architecture,
                "transfer_type": EXP.training.transfer_type,
                "learning_rate": EXP.training.learning_rate,
                "momentum": EXP.training.momentum,
                "mtl": EXP.is_mtl,
                "primary_task_weight": EXP.primary_task_weight,
                "effective_batch_size": EFFECTIVE_BATCH_SIZE,
                "batch_size": BATCH_SIZE,
                "accumulation_steps": ACCUMULATION_STEPS,
                "max_epochs": MAXIMUM_EPOCHS,
                "early_stopping_patience": EARLY_STOPPING_PATIENCE,
                "test_split": TEST_SPLIT,
                "gpu_augment": EXP.aug_magnitude,
                "dataset": PRODUCE_DATASET_PATH.name.split("_", 4)[4],
                "pretrained": EXP.pretrained,
                "class_weighted": EXP.class_weighted,
            },
        )

    # Set accuracy metrics (one per task, per split)
    train_health_accuracy = torchmetrics.Accuracy(task="binary", num_classes=NUM_CLASSES).to(device)
    train_type_accuracy = torchmetrics.Accuracy(task="multiclass", num_classes=produce_dataset.num_produce_types).to(device)
    val_health_accuracy = torchmetrics.Accuracy(task="binary", num_classes=NUM_CLASSES).to(device)
    val_type_accuracy = torchmetrics.Accuracy(task="multiclass", num_classes=produce_dataset.num_produce_types).to(device)


    best_val_health_loss = float('inf')
    patience_counter = 0

    for epoch in range(MAXIMUM_EPOCHS):
        (train_loss, train_health_loss, train_type_loss,
         train_health_acc, train_type_acc) = train_one_epoch(
            model, train_loader, health_criterion, type_criterion, 
            optimizer, device, epoch, train_health_accuracy, 
            train_type_accuracy, EXP.is_mtl, 
            gpu_augment if AUGMENT else None)

        (val_loss, val_health_loss, val_type_loss,
         val_health_acc, val_type_acc) = validate(
            model, val_loader, health_criterion, type_criterion,
            device, epoch, val_health_accuracy, val_type_accuracy, EXP.is_mtl)

        # Step scheduler at each epoch
        lr_scheduler.step()

        # Early stopping on val loss
        if val_health_loss < best_val_health_loss:
            best_val_health_loss = val_health_loss
            patience_counter = 0
            # Save in safetensor format to safety
            save_file(model.state_dict(), f'models/{model_save_name}.safetensors')
            type_msg = f"; Type acc {val_type_acc:.4f}" if EXP.is_mtl else ""
            print(f"Epoch {epoch+1}: New best model saved with Val Loss: {best_val_health_loss:.4f}; Health acc {val_health_acc:.4f}{type_msg}")
        else:
            patience_counter += 1
            if patience_counter >= EARLY_STOPPING_PATIENCE:
                print(f"Early stopping at epoch {epoch+1}")
                break

        # Log metrics to W&B (type metrics only when MTL is active)
        log_data = {
            "loss/train_combined": train_loss,
            "loss/train_health": train_health_loss,
            "loss/val_combined": val_loss,
            "loss/val_health": val_health_loss,
            "accuracy/train_health": train_health_acc,
            "accuracy/val_health": val_health_acc,
            "lr": optimizer.param_groups[0]["lr"],
        }
        if EXP.is_mtl:
            log_data.update({
                "loss/train_type": train_type_loss,
                "loss/val_type": val_type_loss,
                "accuracy/train_type": train_type_acc,
                "accuracy/val_type": val_type_acc,
            })
        if RECORD_WANDB:
            wandb.log(log_data, step=epoch)
    if RECORD_WANDB:
        wandb.finish()

In [ ]:
# Calibration check - measure softmax over-confidence of the trained model.
# The softmax confidence feeds the ripeness grade at inference, so we want to
# know how well calibrated it is before claiming "high confidence = high quality".

if TEST_CONFIDENCE:
    TEST_MODEL = None
    create_calibration_plots(model=model, model_save_name=model_save_name, 
                             val_loader=val_loader, device=device, experiment=EXP, 
                             test_model=TEST_MODEL)